In [ ]:
from pathlib import Path
import polars as pl
from dtale import show
import dtale.global_state as global_state

global_state.set_app_settings(dict(max_column_width=300))

data_dir = Path().absolute() / ".." / "data"
df = pl.read_parquet(data_dir / "interests_clusters/v3/cm0i27jdj0000aqpa73ghpcxf.snappy")
      # .drop("raw_interests", "raw_results")
      # .explode("interests", "interests_quirkiness")
      # .filter(pl.col("interests_quirkiness")))
d = show(df.to_pandas())
d.open_browser()

In [ ]:
from textwrap import dedent


interests_clusters = df

def get_categorization_prompt(cluster_interests: str) -> str:
    return dedent(f"""
    Analyze this list with a few samples from a given category of activities.

    After your analysis, come up with the most granluar possible category description that encompasses each record.
    Avoid providing a description that is too generic, enrich your answer with details on the contents.
                  
    Conclude your answer in one line JSON format: {{ "description": "your answer" }}.
    Think step by step.

    {cluster_interests}
    """).strip()

sampled_df = (
    interests_clusters.filter(pl.col("merged_cluster_label") != -1)
    .group_by("merged_cluster_label")
    .apply(lambda group: group.sample(n=min(len(group), 100)))
)

# Group interests by merged_cluster_label
grouped_interests = (
    sampled_df.group_by("merged_cluster_label")
    .agg(pl.col("interests").alias("cluster_interests"))
    .sort("merged_cluster_label")
)

# Prepare prompts for each merged cluster
prompt_sequences = [
    get_categorization_prompt("\n".join(row["cluster_interests"]))
    for row in grouped_interests.to_dicts()
]

In [ ]:
prompt_sequences[2]

In [ ]:
COL = "merged_cluster_label"

df = df.sort("merged_cluster_label", "date", descending=[False, False])

res = df.with_columns(
        pl.concat_str(
            [
                pl.when(pl.col("interests_quirkiness").eq(True))
                .then(pl.lit("NB:"))
                .otherwise(pl.lit("")),
                pl.col("date"),
                pl.lit(":"),
                pl.col("interests"),
            ],
        ).alias("date_interests")
).group_by(
    COL
).agg(
    # Aggregate the interests with their dates into a list of strings
    pl.col("date_interests").str.concat("\n").alias("cluster_items"),
).filter(
    pl.col(COL) == 81
).get_column("cluster_items")[0]

print(res)

In [ ]:
import umap
import plotly.express as px

In [ ]:
df = df.filter(pl.col("fine_centroid").is_not_null()).group_by("fine_dissimilarity_rank").agg(pl.col("fine_centroid").first())

# Reduce dimensionality of fine_centroid embeddings
reducer = umap.UMAP(n_components=2, random_state=42)
embeddings_2d = reducer.fit_transform(df['fine_centroid'].to_list())

# Create a new dataframe with reduced embeddings and labels
plot_df = pl.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'rank': df['fine_dissimilarity_rank']
})

# Create an interactive scatter plot
fig = px.scatter(
    plot_df.to_pandas(),
    x='x',
    y='y',
    color='rank',
    hover_data=['rank'],
    title='Fine Centroid Embeddings Visualization'
)

# Show the plot
fig.show()

In [ ]:
from sklearn.metrics import pairwise_distances
import numpy as np

# Create some sample data
X = np.array([[-9999, -9999], [1, 1], [-1, -1], [9999, 9999]])

# Calculate pairwise distances using Euclidean distance (default metric)
distances = pairwise_distances(X, metric='cosine')

print("Pairwise distances:")
print(distances)

# Calculate pairwise distances using Manhattan distance
manhattan_distances = pairwise_distances(X, metric='manhattan')

print("\nPairwise Manhattan distances:")
print(manhattan_distances)